## AI per Database Relazionale


In [8]:
import os
import getpass
secret_key = getpass.getpass(prompt='Inserisci la chiave: ')

Inserisci la chiave:  ········


In [9]:
server_name = r'LAPTOP-UDP6N0UL\SQLEXPRESS'
database_name = 'CorsoSQL_2'
utente = getpass.getpass(prompt='Inserisci nome utente: ') #'utente_ai'#
password = getpass.getpass(prompt='Inserisci la password: ')# 'PasswordStrong'

Inserisci nome utente:  ········
Inserisci la password:  ········


In [10]:
from sqlalchemy import create_engine, event, text
from sqlalchemy.engine import Engine
import pyodbc

#ricordarsi di aggiungere in produzione &Encrypt=yes&TrustServerCertificate=no
conn_str = f'mssql+pyodbc://{utente}:{password}@{server_name}/{database_name}?driver=ODBC+Driver+17+for+SQL+Server' 

engine = create_engine(conn_str)

#@event.listens_for(engine, "connect")
#def set_pyodbc_timeout(dbapi_connection, connection_record):
#    dbapi_connection.timeout = 1
#engine.dispose()

# SQLDatabase di LangChain

In [11]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import create_sql_agent
from langchain.chat_models import init_chat_model

# ===== LLM =====
llm = init_chat_model(
    model="claude-opus-4-6",
    model_provider="anthropic",
    temperature=0,
    max_tokens=8000,
    api_key=secret_key
)

# Oggetto database di LangChain
db = SQLDatabase(engine, 
                 schema="dbo")

# ===== AGENTE =====
agent = create_sql_agent(
    llm=llm, 
    db=db,
    verbose=True,    
    max_iterations=50, 
    max_execution_time=1000,
)

## Test 1

In [12]:
query = """
SELECT COUNT(*)
FROM  dbo.Fatture
WHERE YEAR(DataFattura) = 2023
"""

In [16]:
domanda = f"""Ottimizza la query seguente per minimizzare elapsed time, cpu e memory grant.

{query}

Analizza il piano di esecuzione stimato delle query prima di proporre varianti identificando potenziali colli di bottiglia.

"""

In [17]:
# Domanda in linguaggio naturale
# Anche se si è generato un errore, è presente tutta la 
response = agent.invoke({"input": domanda})



> Entering new SQL Agent Executor chain...
I need to first understand the database structure and then analyze the query execution plan to identify bottlenecks before proposing optimizations.

Thought: Let me start by listing the tables and then examining the schema of the Fatture table.

Action: sql_db_list_tables
 Clienti, Clienti2, ColonnaMista, Corrieri, DataOra, Fatture, Fatture2, FattureProdotti, Fornitori, Prodotti, Test1, Test2

Action: sql_db_schema
Action Input: Fatture
CREATE TABLE dbo.[Fatture] (
	[IdFattura] INTEGER NOT NULL IDENTITY(1,1), 
	[IdCliente] INTEGER NOT NULL, 
	[IdFornitore] INTEGER NOT NULL, 
	[DataFattura] DATE NOT NULL, 
	[DataTerminePagamento] DATE NOT NULL, 
	[DataPagamento] DATE NULL, 
	[Spedizione] DECIMAL(18, 2) NULL, 
	[IdCorriere] INTEGER NULL, 
	[DataArrivoRichiesta] DATE NULL, 
	[DataArrivoEffettiva] DATE NULL, 
	CONSTRAINT [PkFattura] PRIMARY KEY ([IdFattura]), 
	CONSTRAINT [FkClienti] FOREIGN KEY([IdCliente]) REFERENCES dbo.[Clienti] ([IdCliente]

In [18]:
print(response["output"])

**

## Analisi della Query Originale

```sql
SELECT COUNT(*)
FROM dbo.Fatture
WHERE YEAR(DataFattura) = 2023
```

### Struttura della Tabella
- **Tabella:** `dbo.Fatture` — **90.300 righe** totali, di cui **7.224** con `DataFattura` nel 2023
- **Indice Clustered:** `PkFattura` su `IdFattura`
- **Indice Non-Clustered:** `ix_windowfunction` su `(IdCorriere, DataFattura)`

---

### 🔴 Colli di Bottiglia Identificati

#### 1. **Funzione `YEAR()` sulla colonna — NON SARGABLE**
L'uso di `YEAR(DataFattura)` applica una **funzione scalare** sulla colonna, rendendo il predicato **non-SARGable** (Search ARGument able). Questo significa che:
- SQL Server **non può fare un Index Seek** sulla colonna `DataFattura`
- È costretto a fare un **Index Scan o Table Scan** leggendo **tutte le 90.300 righe**
- La funzione `YEAR()` viene calcolata **riga per riga**, aumentando il consumo di CPU

#### 2. **Indice esistente non ottimale**
L'indice `ix_windowfunction (IdCorriere, DataFattura)` ha `DataFattura` c

## Test 2

In [19]:
query = """
SELECT COUNT(*)
FROM  dbo.DataOra
WHERE CAST(mydate as datetime) + 
	  CAST(mytime as datetime)
	     >= CAST('19800301 15:00:00' AS Datetime)
AND CAST(mydate as datetime) + 
    CAST(mytime as datetime)
	     < CAST('19800401 15:00:00' AS Datetime); 
"""

In [20]:
domanda = f"""Ottimizza la query seguente per minimizzare elapsed time, cpu e memory grant.

{query}

Analizza il piano di esecuzione stimato delle query prima di proporre varianti identificando potenziali colli di bottiglia.

"""

In [22]:
# Domanda in linguaggio naturale
# Anche se si è generato un errore, è presente tutta la 
response = agent.invoke({"input": domanda})



> Entering new SQL Agent Executor chain...
I need to first understand the database structure and the table schema to optimize this query effectively.

Thought: Let me start by listing the tables and then examining the schema of the DataOra table.

Action: sql_db_list_tables
Now let me examine the schema of the DataOra table to understand the column types and indexes.attureProdotti, Fornitori, Prodotti, Test1, Test2

Action: sql_db_schema
Action Input: DataOra
CREATE TABLE dbo.[DataOra] (
	[MyDate] DATE NULL, 
	[MyTime] TIME NULL
)

/*
3 rows from DataOra table:
MyDate	MyTime
1985-03-07	22:26:31.204000
1983-11-24	07:16:24.273000
1981-12-29	22:21:15.248000
Now let me analyze the original query's execution plan to understand the bottlenecks.

Action: sql_db_query
Action Input: SET SHOWPLAN_TEXT ON; GO; SELECT COUNT(*) FROM dbo.DataOra WHERE CAST(mydate as datetime) + CAST(mytime as datetime) >= CAST('19800301 15:00:00' AS Datetime) AND CAST(mydate as datetime) + CAST(mytime as datetime)

In [23]:
print(response["output"])

# Analisi e Ottimizzazione della Query su `dbo.DataOra`

## 📊 Contesto
- **Tabella**: `dbo.DataOra` — **1.000.000 righe**
- **Colonne**: `MyDate` (DATE), `MyTime` (TIME)
- **Indice esistente**: `ix_data_ora` NONCLUSTERED su `(MyDate, MyTime)`
- **Risultato atteso**: **10.325 righe**

---

## 🔴 Problemi della Query Originale

```sql
SELECT COUNT(*)
FROM dbo.DataOra
WHERE CAST(mydate as datetime) + CAST(mytime as datetime)
         >= CAST('19800301 15:00:00' AS Datetime)
AND CAST(mydate as datetime) + CAST(mytime as datetime)
         < CAST('19800401 15:00:00' AS Datetime);
```

### Colli di bottiglia identificati:

1. **Predicati NON-SARGable**: L'espressione `CAST(mydate as datetime) + CAST(mytime as datetime)` avvolge le colonne in funzioni e operazioni aritmetiche. Questo **impedisce completamente** a SQL Server di utilizzare l'indice `ix_data_ora` tramite **Index Seek**. Il piano di esecuzione sarà forzato a eseguire un **Index Scan (o Table Scan) completo** su tutte le 1.000.000 